In [ ]:
import time
from typing import Annotated, TypedDict, Literal
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

# ==========================================
# 1. 定义状态 (AgentState)
# ==========================================
class AgentState(TypedDict):
    # messages: 所有消息的历史记录 (Human, AI, Tool)
    # 使用 add_messages 作为 reducer，表示追加而非覆盖
    messages: Annotated[list[BaseMessage], add_messages]
    
    # 自定义业务状态
    model_call_count: int          # 记录模型调用次数
    user_id: str                   # 用户ID
    start_time: float              # 流程开始时间戳
    end_time: float                # 流程结束时间戳

# ==========================================
# 2. 定义工具 (Tools) - 为了演示 ToolMessage
# ==========================================
@tool
def get_current_time() -> str:
    """获取当前系统时间"""
    return f"当前时间是: {time.strftime('%Y-%m-%d %H:%M:%S')}"

tools = [get_current_time]

# ==========================================
# 3. 初始化模型 (Model)
# ==========================================
# 绑定工具，让模型知道可以调用什么
llm = ChatOllama(model="qwen3.6:latest",base_url="http://192.168.8.21:11434").bind_tools(tools)

# ==========================================
# 4. 定义节点逻辑 (Nodes) - 这里是我们“观察”的核心
# ==========================================

def node_agent(state: AgentState):
    """核心代理节点：负责调用 LLM"""
    print("\n" + "="*50)
    print("🔍 [节点: Agent] 正在执行...")
    print("-" * 50)
    print(f"📥 当前 State 内容: {dict(state)}")
    
    # 记录调用次数
    current_count = state.get("model_call_count", 0) + 1
    
    print(f"🤖 [ModelRequest] 即将发送给模型的 Messages:")
    for msg in state["messages"]:
        print(f"  - {type(msg).__name__}: {msg.content}")

    # 调用模型
    response = llm.invoke(state["messages"])
    
    print(f"✨ [ModelResponse] 模型返回内容:")
    print(f"  - 类型: {type(response).__name__}")
    print(f"  - 内容: {response.content}")
    print(f"  - 工具调用: {response.tool_calls}")

    # 返回值会被自动合并到 State 中
    return {
        "messages": [response],
        "model_call_count": current_count
    }

def node_tool_executor(state: AgentState):
    """工具执行节点：这里我们使用 LangGraph 预定义的 ToolNode 逻辑，但手动包装一下以便打印"""
    print("\n" + "="*50)
    print("🔧 [节点: Tools] 正在执行...")
    print("-" * 50)
    
    # 手动实例化 ToolNode 并执行
    tool_node = ToolNode(tools)
    result = tool_node.invoke(state)
    
    print(f"🛠️  [ToolMessage] 工具执行结果:")
    for msg in result["messages"]:
        print(f"  - {type(msg).__name__}: {msg.content}")
    return result

def node_start_timer(state: AgentState):
    """入口节点：仅用于记录开始时间"""
    print("\n" + "="*50)
    print("🚀 [节点: Start] 初始化流程...")
    print("-" * 50)
    start_time = time.time()
    print(f"⏱️  记录开始时间: {start_time}")
    return {"start_time": start_time}

def node_end_timer(state: AgentState):
    """出口节点：计算耗时"""
    print("\n" + "="*50)
    print("🏁 [节点: End] 流程结束...")
    print("-" * 50)
    end_time = time.time()
    duration = end_time - state["start_time"]
    print(f"⏱️  结束时间: {end_time}")
    print(f"⌛ 总耗时: {duration:.2f} 秒")
    print(f"🔢 模型调用总次数: {state.get('model_call_count', 0)}")
    return {"end_time": end_time}

# ==========================================
# 5. 定义路由逻辑 (Conditional Edge)
# ==========================================
def should_continue(state: AgentState) -> Literal["tools", "node_end_timer"]:
    """决定下一步是调用工具还是结束"""
    messages = state["messages"]
    last_message = messages[-1]
    
    # 如果是 AIMessage 且有 tool_calls，则去工具节点
    if isinstance(last_message, AIMessage) and last_message.tool_calls:
        print(f"\n🧭 路由决策: 发现工具调用，转向 Tools 节点")
        return "tools"
    
    # 否则结束
    print(f"\n🧭 路由决策: 无工具调用，转向结束")
    return "node_end_timer"

# ==========================================
# 6. 构建图 (Graph)
# ==========================================
builder = StateGraph(AgentState)

# 添加节点
builder.add_node("node_start_timer", node_start_timer)
builder.add_node("node_agent", node_agent)
builder.add_node("tools", node_tool_executor)
builder.add_node("node_end_timer", node_end_timer)

# 定义边
builder.add_edge(START, "node_start_timer")
builder.add_edge("node_start_timer", "node_agent")

# 定义条件边 (Agent -> Tools 或 End)
builder.add_conditional_edges(
    "node_agent",
    should_continue,
    {
        "tools": "tools",
        "node_end_timer": "node_end_timer"
    }
)

# 定义工具执行完后的回流 (Tools -> Agent)
builder.add_edge("tools", "node_agent")
builder.add_edge("node_end_timer", END)

# 编译图
graph = builder.compile()

# ==========================================
# 7. 执行与测试
# ==========================================
if __name__ == "__main__":
    print(">>> 启动 Agent 透视系统 <<<")
    
    # 初始输入
    initial_input = {
        "messages": [HumanMessage(content="你好，请问现在几点了？")],
        "user_id": "靓仔",
        "model_call_count": 0
    }

    # 执行
    result = graph.invoke(initial_input)
    
    print("\n>>> 最终 State 快照 <<<")
    print(result)

>>> 启动 Agent 透视系统 <<<

🚀 [节点: Start] 初始化流程...
--------------------------------------------------
⏱️  记录开始时间: 1777293105.405631

🔍 [节点: Agent] 正在执行...
--------------------------------------------------
📥 当前 State 内容: {'messages': [HumanMessage(content='你好，请问现在几点了？', additional_kwargs={}, response_metadata={}, id='c8e4fe7e-7876-40d4-9794-3fc1398f0dc6')], 'model_call_count': 0, 'user_id': '靓仔', 'start_time': 1777293105.405631}
🤖 [ModelRequest] 即将发送给模型的 Messages:
  - HumanMessage: 你好，请问现在几点了？
✨ [ModelResponse] 模型返回内容:
  - 类型: AIMessage
  - 内容: 
  - 工具调用: [{'name': 'get_current_time', 'args': {}, 'id': 'e9c7abcf-e352-4716-8c7e-1f28ea4b05a4', 'type': 'tool_call'}]

🧭 路由决策: 发现工具调用，转向 Tools 节点

🔧 [节点: Tools] 正在执行...
--------------------------------------------------
🛠️  [ToolMessage] 工具执行结果:
  - ToolMessage: 当前时间是: 2026-04-27 20:31:57

🔍 [节点: Agent] 正在执行...
--------------------------------------------------
📥 当前 State 内容: {'messages': [HumanMessage(content='你好，请问现在几点了？', additional_kwargs={}, 